<div dir="rtl">

# ۳ – ارزیابی ساده‌ی RAG و جمع‌بندی نتایج

در این نوت‌بوک، مجموعه‌ای از سؤال‌های ارزیابی را آماده می‌کنید، سیستم RAG خود را روی آن‌ها اجرا می‌کنید، کیفیت پاسخ‌ها را بررسی می‌کنید و در پایان نتایج را به‌صورت کوتاه جمع‌بندی می‌کنید.

</div>


In [ ]:
# TODO: import های لازم را بنویسید
# مثال:
import json
from pathlib import Path

In [ ]:
# TODO: فایل 'evaluation/questions_template.jsonl' را باز کنید و حداقل ۱۵ سوال و پاسخ مرجع در آن بنویسید
# می‌توانید این کار را دستی (ویرایش فایل) یا با همین نوت‌بوک انجام دهید
file_path = "evaluation/questions_template.jsonl"

qa_pairs = [
    {
        "question": "پیروزی تیم فوتبال تبریز در لیگ برتر چگونه بود؟",
        "reference_answer": "تیم فوتبال تبریز در لیگ برتر مقابل حریفان خود به برتری رسید و با گل‌های متعدد پیروز شد."
    },
    {
        "question": "جدیدترین اخبار اقتصادی ایران چیست؟",
        "reference_answer": "اخبار اقتصادی ایران شامل تغییرات نرخ ارز، بازار سکه، و سیاست‌های اقتصادی دولت می‌باشد."
    },
    {
        "question": "چه رویداد سیاسی مهمی اخیراً رخ داده است؟",
        "reference_answer": "یک رویداد سیاسی مهم اخیر برگزاری نشست بین‌المللی و تصمیم‌گیری‌های کلان کشور بود."
    },
    {
        "question": "اخبار ورزشی امروز شامل چه تیم‌هایی است؟",
        "reference_answer": "اخبار ورزشی امروز شامل تیم‌های فوتبال، والیبال و بسکتبال از لیگ‌های داخلی و خارجی است."
    },
    {
        "question": "آخرین خبر درباره بازار ارز و سکه چیست؟",
        "reference_answer": "بازار ارز و سکه شاهد نوساناتی بود و نرخ دلار و سکه طلا تغییراتی را تجربه کردند."
    },

    {
        "question": "آخرین وضعیت تیم ملی فوتبال ایران چگونه است؟",
        "reference_answer": "تیم ملی فوتبال ایران برای مسابقات جام جهانی آماده می‌شود و تمرینات فشرده‌ای دارد."
    },
    {
        "question": "جدیدترین اخبار فناوری در ایران چیست؟",
        "reference_answer": "اخبار فناوری شامل معرفی اپلیکیشن‌ها و پروژه‌های نوآورانه در ایران است."
    },
    {
        "question": "چه خبرهایی از صنعت نفت ایران منتشر شده؟",
        "reference_answer": "اخبار صنعت نفت ایران شامل تولید، صادرات و قیمت جهانی نفت می‌باشد."
    },
    {
        "question": "وضعیت کرونا در ایران چگونه است؟",
        "reference_answer": "تعداد مبتلایان و واکسیناسیون در ایران تحت کنترل بوده و اخبار رسمی وزارت بهداشت منتشر می‌شود."
    },
    {
        "question": "اخبار مربوط به سیاست خارجی ایران چیست؟",
        "reference_answer": "اخبار سیاست خارجی شامل مذاکرات بین‌المللی و روابط ایران با دیگر کشورهاست."
    },
    {
        "question": "چه خبرهایی از لیگ برتر والیبال ایران داریم؟",
        "reference_answer": "لیگ برتر والیبال ایران با رقابت تیم‌ها و نتایج تازه در حال برگزاری است."
    },
    {
        "question": "تازه‌ترین تحولات در بازار مسکن چیست؟",
        "reference_answer": "بازار مسکن تغییرات قیمتی و عرضه و تقاضا را تجربه می‌کند."
    },
    {
        "question": "چه اخبار فرهنگی در ایران منتشر شده است؟",
        "reference_answer": "اخبار فرهنگی شامل جشنواره‌ها، نمایشگاه‌ها و فعالیت‌های هنری می‌باشد."
    },
    {
        "question": "وضعیت هوا و آب و هوا در ایران چگونه است؟",
        "reference_answer": "پیش‌بینی هوا شامل دما، بارندگی و شرایط جوی مختلف در شهرهای ایران است."
    },
    {
        "question": "آخرین اخبار ورزش بانوان چیست؟",
        "reference_answer": "اخبار ورزش بانوان شامل مسابقات، نتایج و دستاوردهای ورزشکاران زن ایران است."
    }
]

with open(file_path, "w", encoding="utf-8") as f:
    for qa in qa_pairs:
        f.write(json.dumps(qa, ensure_ascii=False) + "\n")

print(f"{len(qa_pairs)} سوال و پاسخ مرجع در فایل '{file_path}' ذخیره شد.")

In [ ]:
# TODO: تابعی برای خواندن سوال‌ها از فایل jsonl بنویسید
# خروجی: لیستی از دیکشنری‌ها با کلیدهای id, question, reference_answer, doc_ids
def load_questions(jsonl_file_path):
     
    questions = []
    with open(jsonl_file_path, "r", encoding="utf-8") as f:
        for idx, line in enumerate(f):
            item = json.loads(line.strip())
            questions.append({
                "id": idx + 1,
                "question": item.get("question", ""),
                "reference_answer": item.get("reference_answer", ""),
                "doc_ids": []  
            })
    return questions

jsonl_path = "evaluation/questions_template.jsonl"
questions_list = load_questions(jsonl_path)
 
for q in questions_list[:3]:
    print(f"ID: {q['id']}")
    print(f"Question: {q['question']}")
    print(f"Reference Answer: {q['reference_answer']}")
    print(f"Doc IDs: {q['doc_ids']}")
    print("-" * 50)

In [ ]:
# TODO: برای هر سوال، تابع answer(question, embedding_model) را با سه حالت embedding از نوت‌بوک 02 صدا بزنید
# نتایج (پاسخ مدل و chunk های بازیابی شده) را ذخیره کنید
def run_evaluation(questions_list, 
                   index, 
                   df_chunks, 
                   semantic_model=None, 
                   tfidf_vectorizer=None, 
                   alpha=0.7, 
                   beta=0.3, 
                   top_k=5):
     
    eval_results = []
    
    for q in questions_list:
        question_text = q["question"]
        q_id = q["id"]
        ref_answer = q["reference_answer"]
         
        sem_top_chunks = retrieve(
            question=question_text,
            top_k=top_k,
            embedding_model=semantic_model,
            tfidf_vectorizer=None,
            index=index,
            df_chunks=df_chunks
        )
        sem_answer = "\n\n".join([c["text"] for c in sem_top_chunks])
        sem_chunk_ids = [c["chunk_id"] for c in sem_top_chunks]
         
        lex_top_chunks = retrieve(
            question=question_text,
            top_k=top_k,
            embedding_model=semantic_model,   
            tfidf_vectorizer=tfidf_vectorizer,
            alpha=0.0,   
            beta=1.0,    
            index=index,
            df_chunks=df_chunks
        )
        lex_answer = "\n\n".join([c["text"] for c in lex_top_chunks])
        lex_chunk_ids = [c["chunk_id"] for c in lex_top_chunks]
         
        hybrid_top_chunks = retrieve(
            question=question_text,
            top_k=top_k,
            embedding_model=semantic_model,
            tfidf_vectorizer=tfidf_vectorizer,
            alpha=alpha,
            beta=beta,
            index=index,
            df_chunks=df_chunks
        )
        hybrid_answer = "\n\n".join([c["text"] for c in hybrid_top_chunks])
        hybrid_chunk_ids = [c["chunk_id"] for c in hybrid_top_chunks]
         
        eval_results.append({
            "id": q_id,
            "question": question_text,
            "reference_answer": ref_answer,
            "results": {
                "semantic": {
                    "answer": sem_answer,
                    "chunk_ids": sem_chunk_ids
                },
                "lexical": {
                    "answer": lex_answer,
                    "chunk_ids": lex_chunk_ids
                },
                "hybrid": {
                    "answer": hybrid_answer,
                    "chunk_ids": hybrid_chunk_ids
                }
            }
        })
    
    return eval_results

eval_results = run_evaluation(
    questions_list=questions_list,
    index=index,
    df_chunks=df_chunks,
    semantic_model=semantic_model,
    tfidf_vectorizer=tfidf_vectorizer,
    alpha=0.7,
    beta=0.3,
    top_k=5
)

print(f"نتایج {len(eval_results)} سوال ذخیره شد.")

In [ ]:
# TODO: یک معیار ساده تعریف کنید (مثلاً قبول/رد دستی برای هر سوال)
# درصد سوال‌هایی که جواب قابل قبول دارند را حساب کنید


In [ ]:
# TODO: ۲–۳ نمونه پاسخ خوب و ۲–۳ نمونه پاسخ بد را چاپ کنید و به صورت متنی توضیح دهید چرا خوب/بد هستند


<div dir="rtl">

## خلاصه برای گزارش

در این بخش چند پاراگراف بنویسید و خلاصه کنید:

- از نظر شما، سیستم تقریباً به چند درصد سؤال‌ها پاسخ «قابل قبول» می‌دهد؟
- بزرگ‌ترین محدودیت‌ها و خطاهای تکرارشونده کدامند؟
- اگر زمان و منابع بیشتری در اختیار داشتید، چه بهبودهایی برای نسخه‌ی بعدی سیستم پیشنهاد می‌کنید؟  
  (برای مثال استفاده از مدل بهتر، بهبود روش چانک‌کردن متن‌ها، افزودن مراحل پیش‌پردازش یا تغییر شیوه‌ی تولید پاسخ.)

این متن می‌تواند مستقیماً در گزارش نهایی پروژه (به صورت PDF یا Markdown) استفاده شود.

</div>
